# Laya Wikiracing: base versus fine-tuned

A fixed Wikispeedia graph, target-disjoint splits, and identical 60-race evaluation. The notebook runs base Laya before training, then full supervised fine-tuning on one T4. It records decision accuracy, race success, and GPU model latency separately. Source: https://snap.stanford.edu/data/wikispeedia.html


In [ ]:
!pip -q install laya==0.3.5 transformers==4.57.6 safetensors==0.6.2 numpy==2.0.2


In [ ]:
import os, base64, io, tarfile, shutil
from pathlib import Path
os.environ['USE_TF']='0'
os.environ['USE_FLAX']='0'
payload = 'H4sIAI6st2oC/+08a3PbRpL5zF+BxdXdAjYEkZIo29zl3mkTJXGtN1bJ9ubqWCzUCBiSOIIAggEoyT799+vumQEGICnbycZ3lWDKtsh5dPf09HtG9o/94/+4Ynffcxbx4qtfpQ1lO/RzODw9az5j/2h4Mjr5yrr76gu0SpSsAPRf/T7byXNrU8YbPh09ezE8Oxk+G5/645Phc/g8+Kpvv/l2G6/jgoX8OAjiNC6DwM/vfw39Pz+TOv7sfCx1/cTQ+fNRR//PRjBsDb+k/q9YEa/zw/Ng2mLx2zt/27Z/BBEQOedRzCwUBStiJbNYGlmbLOKJVZVxEpcxFz5M7o3Cb6v5vf/v/X/j/0fj8xN/fHo2Pn9+2qv678n/LwuWr34F5/9x/392pvtq/T8dD096//+F/P+rjEVWueIWK8JVvOWRZUYEJBcUDYRZKsqiCksr4UuWqOggXGVxqGKDeJNnRWn9t8hS/XnFxCqJb/TXDStX+nPBB4si2wDcJOFhGQN4Sw1F/KdKjeawAgDokSsEMPju+uLq++Dq4u331pS6nCBYxAkPAtcvuMiSLXdcP2cFT0sxG82tY8vGqOa4I+5IqT0YDCK+sBJgQ0DdDuKcNjjcycCCVvCyKlLanY+ThUOYcTJihdUlvysd11UAS15shFPGZcLbEAQvnYL7iziNWJI4hT1jR++HRy/mT23PovkA/5YXAMq1jqwPNhwOjNjZAv+Fo8AfcUpf7AeFTayAPUksSof24FlhVeD2ASIrlhx+JvEmLqejE0UMHNg1S9dWlib36kSzqlxmcbqEqelaeNZtXK6gj06+SkociQADSyFILDMFmE4eARLlwpOL4VyIjpktu+25pzto3J7TGrZlccJuEg7zqX+myJbD8UIhseK0mSvpxyY4Sg5I7NSayYlz66k121qLrLC2rUUIa2v9YaoAzmcT4gfwdzSv4ekDAlbyyNHQPWvN76cJ29xEzNpOtEz7YsVOxufOwv6giH6YfJDQ4cP2wfZ5GoKOwDH6UbzkgmSDOEWTAhIQIN0QFKG3ISeChPBlwXnDTd0D/KMZdPRhVnBn6zZ8KbtQtwogtmzLi4TlyHGeOqX1by16XNCVDbtzRp4a/p/2cAPnX6zvqxsBVoNbleCLKiGuw0wSFJSbbGGlrCiyW5CWPA5BNm6gN8qsNMMfmzhlJfe73HfOrCc1kU+toT8aQgdaDlCL5Sh3NA9wV551tHUlKwxhUAdYH748QeKTB2gAtuDTt0XFXSUG88EXOn6prGDdBBq8WldBqWMwUFy0LQWo/n3O7YllSzMrFV9aYbSXOPLjKg5XpDywdVaUcYjCLsii3wAaKwWrZImS53AKt6yQtl6plZr/7wg4LGI44pgB0A8L0tLgQ/wA3zqKDGynk449qWI8rTa8AOIdYxsPtV0qceSQTdrGApBGU2BNZ+NqaqAo3EOGthRAupLQw3OVVsHUgocGWOTg7MD+aHNllSegW5JMd3Z0OpnP9daUICmnQf+qXcRw8hvUArBLMwktQGgdTFLqcLTy4Jji5eomK0Sbp22jaeh4TWK9sBkzaYDt+CzPeRo5lWvyWE9Qu9GWXQRl5ugxfVIKbz0HNqZkfWINH2gMZLpCQ0W+22nbsdsVOGc5oyFyC5Opy88zYPIC9KO1uQo3Z+yis70FTEA7ApNqstpTWhTPqjkSV3/doqcY7cyXBO3lV732t5QY9fl/n/+b+f/p6Qv/7MXp+PnzZ33+/3vK/8uCxen/Rf5/OhybtkDn/6d9/v+F8v8rFhcQOt4wwSnPh0geQvk45UdllXKLb1lSMQw3IVWk2NEsD0Q8hNgIB5hYHyoCmFl/Jur8H5BlG/0NjdAjGX8mIJzexgXk3lWOIabz7s1l8PbbqT2E2BU/f/vq4j/lt7ev/3b5w8v/urx+A+n79cWrV5evXr75+9ResERwG2JwBRlirPzeYsJK85qKrAhXkoxVtVxC4LEA5QhWVU2OSFkOuXYZRNltimUAOVuwBS95CqmD8AmInk9lBaxNeDBny+mjXAIalwoIdCDQqwsfF1WZvc3WPI3f80JOS9g989kSglY9CWDcQZCmZgWQnS/ipTEZIqYNnIiafVPFSRRQscZTXwSGaBDJDOQibQN8WesxCVdxe11c8OrUxZOR/WBw/fr120+rwgy+uXh7AVNpRbciYw9ev6vhGKcNQaRjw9DVu7fBNy+vMVE5XrPlMuHHt1mxhhM6xk0f1XAgybq+/MfLNy9f/wDg7FE45lE4esaesTB6/mw4ihbni/D0bBQOI/B5i1F4fhK+eH5i10E9bBtSVuGIPInLdloyM4o/EJFzlwJV/IRhqEP7O7YgI6S1D1RfSuxWdcinIVwiHHeukMpMEdE69blCoprdQsq0WCoafoLtQNfM1kegaigiXPENw4jcLiGd+Wkmk8a5zBVlTytpnKtkTw7Vad9cxvFxBHl6nkHCQxW5aUdmOgTObBIDhCnp8KxRAEYc/xLxM3uFm9+wuyDhQLKrCztYXQBcrvUXOa2eYSFLYbCmwcWyDfYQvibJbKVD4L7Afv0DbBW/LIqsgLwcEvw0ZFgRAIgblpCuRbUEQ5qLAP8YR3+cP9huO/2MI+AO4YNPxMoIWUnMAVKLNSgtfK9phN6E3fBEr5Jf5l4rwdBJrZ6kvzYprB5R3+pU84aV4Qr3j2fDIg/s7jYOdV0xtWr+CJVwxVG5gl6s5NDIjDYwlwJboLQ2k1UJtz1d73H/EgCGJSY0dD76C8dJPYnU1QSiEE7ljCRLl3IdK8FIki/Rq0GQMhEk8ZqTNNAshbs9h1AoUt1D8LMcgYO0ifUnr77JssStM/GYxLqdg9O+G1EDOmcwbVKLpGTtvMYoHYE5Vq+tGXAIQpOSKi60JzbHsh9dM97U+xqePAYLy2EtW3cHjsxRckasuSMjRypQb8PTZHomHq/F+nTPadWmL4fYI5aGyVFOSoq5YWdawk6TfIxKHK2zWJzGUstcyT6IvkQVpwteoNUi/+d0qicY+VH9ABzxkjtDr1Eiz3rutosJ4apK19ICixmtnMj1T63n89ZMAz8Ddx4yAR4srCJmt/mwAD9Sjs7d3aJFki3jElgQoEridp0n0gAQEQZrwLlGMhII4sYouC2ApL3oSd7jVt/HuYYisUgywC+FeVVX4cwm2atLInss48ft3p5mS+yw7P1sr0SC8NFthvvQssySGiU7ixir0psctbQquGPIktoIFUCnliM3CW5wYcPZjvxho/AkAWnug0sWOQQQztAfe9bYB3F4MTL4kWQCLdLQHw46rEUABuo2C9/DGoDOBCsKdi+3qfY+1/IA40Tg+RlW38tBWxYA7dOpJXeAdGZLFkX8LoeoIqqA3vd4SfR+ZrLf0H70tAjiz8SL2bBTRtMMwjmeQlK6LZbTupHWVx3wBxteFnEoTKaDZDanoVgXZuDjQsQh+XZTxLww+EjRa8RVUXHWlEQfY+zPYGpDWA0ll1CAlc57YOF+3jaclN4D1sRp6QDyYonu0pygtgqnpedOW1phzKw3rRVLiWjuS6DeARAGOgjoyMcD3dLOUtDkun+SI22BaHkWeQa1UDlODtunRa715Il1AkFqtdE747Rn7MC/Izqbhn5PFrbNUwSJY9afp0YfCh8eAhJozARkHTPBbgSK+Iaz1JkZ638mTohdjvYYohrDMsuiHdjU+Yng3Q50hAUR8I22tC2zAnYH/ozc2eRoBFK6f2w0mXeBIvL03tlDwEf54nai2hTsLUmJYSgx9szyUcBCCEdZeA9TtCAf705ux7MkSLBACtTxPtggPsFoGNzEiFvKG3QR1kYlYcj4Vt+vYOqtzYyKDYyoALIGdaesPZ9HS9pmyLNWWR7Iu+/n//QQgt5LUgSxc/vQuevC0AEmUb5UUMgvv+pIv7VW3TchQa378Fr1Y7wTMQxmO6zheRPV1Lvf49pBshR4NDOKToCA1yrte6eajMlex34DCe56Z6TJ1NCAfOR9grsbeIADmOpUzKbcHqRGppuTR+8V3YPhR5M7Tx67hMUEuUkzJ8aQEe8MD2Np8ryasibBU3fEO4tlEQAP/SPlgJ2VUlIxxPTFfRquigyXOLsTSfYIBZbbfNCRRRBmVVriY5Od2b88kH00mJ2pDc8/M6D9rB0rTam97J59g+9TfAGHZI2Gw+EuGAw45EZmQ5VCGeIybwfRPhUWnUNhR8dAAOQG1Axji/dNbDHfWaVMg96QAuJ+km5/qvJ2Q35psiZdA9YVaejB2i18V0Q+mgXYogrBaIpGSRpCARLYLaG8lb59x7c6HwGJRga0OlCrFcGtXpmyQPoa3gcbnKREROUZKCBkWEuj7iHdQ50tFDMTwlz7DKqWzYqddXAUsEJvdz7f75PlZDQ9t7Jsh534UXUFhTR8dbfyt3pd2zNjgEP7DSBUVwDrFEJGP0CTZEhDME6EqAYppu3wRHDrhyzl+6DzO9zPJyM5Iia0j+LzMTd8D/Lx0MQHeoYvO+KEk5KDxRkPMY5uLXkxfnzJi7Hb5WRJvh2FiRitg5MNi1MdDcibDF9wHjlnJxCAA+ydPmmyNiytWBLo7kFjzAQvAzAcQbnCarFwzupKKbpjw+DFIqgfNjk7FdBrsGqwF1kDtf9GlXLr63ffXFjfXb2DPfxU4W2PKne+fvfW36yjuHBUlZ4eRXkWvwNHHWRr+UZKPTTbyuxHUiK/SncwGSpwZN4DAIf+vntPAnOzFOiO0zTb0n2SoMq9TS+yKKec6sJ9cwQsSbLbIMd6U5GK6cyWsZtx3YJBQZEEdEOi7kLku04MPsnDFMed7trdHD/RhbK9dypOvSFVKF3gi57uA9B60iFCOq9DFSR9j6Xq3tPx6ARiVaNWPh29gB62yQPpbO1FPjq3W8HtdDbysZyBf+etkeDmPpAFOTH9oE2b3h5sonXR5OM1UACBO10Cg2Tu7KteaqsN0C0GPQNK2YZPrNlHYhYzpW+uWHDtjo/DmThA1UabSKLnaSyJbwoSHDpDsCK2K6OovEC3aV9dX15dXF9+A8Mf1vxeOQ+8EZDooc+z6DuCljvwwbNshOM+mFpPdxG0C5SuuijeFGr1boQBiAADpNZOZS1xkVRiZeiSvGebmk+OVR7CZBnEEDF9qWRjuhLQhINSJV9kT80rP4eyJKUHeJhTURa756smwOm6ZoYke30lzAXXqVCYbfKYHu1+i3eqxhLaEsXmAWaBTn396ezFu6vNLtpsgTlfaXBMzmvq0bKUxATHi7QARANI2VNJlkcza8nOfF9puQ0P9QgAdsuLJr7OCsQtHqeBBPYTkOsCm6C3dJ1iWxuh1ya5A0gL089Po/dAd/CWFqRRD2lZvMVbRCmMJLwRhMACQsd6N1gl2NmiJ/O61iB1PHj09jktpydKIpWO//XizeWrlz9cgl7+PDwf1hCeSnOg3q+2MWuDgI53jRePOgB4eNijyW09AS5GMVn/FQ/XeQYUQ2QZ8JRc9YHR9S2EvGCk7UoACRwmQOiAQTWp1kNTqKakXiJkIURQoP14uQ6WEnyjMCMBLF+SnxdoXaLAIVCqnCdpxUg1l6bWMyGj4Y0CEyxFID4F/QIzQkcbC9926b1IB5s050jdZ+KAKOfz8aCP2yifJiMT6vEvIrb50Zl9sAkNHryChgEhlqxO+NEYpMyYgCTr0TGOgrLe4tvaEgWJ3U/94UjxMM/CFekH5LGBAPTgp8Ow2gARp551Bn+Uwy0x1it5jnooV+mn7CGPE6f5hA6mthTk8uYY4DcY8BvhcOtL/6hKdvadFEE95L+iN+uvrp2aS56l3rEjURMLEh/HoXrRU2tkvPxHVTOJf2L5w3MzMsZ5Qwg7WrOOCKoBpgPixZmr/YsAE2rQDkGO/x2c6RvqVkUGOXMFkWhW3Delro/XMFDiiNlNFUzy3lAR5VAKiuIHRsmJalXtgwD/h+m8EWjL0H5VLRYJN27dWwJJRXmSVAdD+zILUkhnDNuhb3d4p46na3ielS0WQv7ui3EXveeu0pCSx24tJbiJgvrUWPb/+gLTuHyTeNPUX1QpeVmW+GEBYwGaTTyl9qWm174hnxX1hUiTeRJJc417upeEVhIWiwX+Vwmc7sz2XZjuycN+wAgKF9FO7DZ0qQw+/SCgtar7NyxE/xB1KltSbHSBRm6WqNkh29Duf9U2amoNscq7Rw6svxgPSXZ3pgitUvoQNGblUIEOTgr/9wDhh0mcS1+UZsUmkGFIy3sZt7L7eAO7MND9qSZF5lDUoa0ezYWez9PEffwajYfErV1O1BnH6++uL9+8AcWQBucp3uNoEIaOWsfHLW9RgMGK6iqJPM7ZEeCb4K80ne2JNAxb2BTmCCn6tga5rM2QlHWLMRKNq2/Uib36SaJDcRGk9SUDNXX9FUsWdT0T4v8yXlZZpdMbHTqp8L0J990mn8IgXkeL+8L8QYeXl1evv/4eGKk2OaNLsi4XyCJj1CQ4ECUfIT1W0VWli8QyPGAtKZ6SokGnogzBbnkPQT7MU3qHD2Gjf1aaIYEdyDFqTObcX55dSDiPphYGKs+g0Vz/yzOKLlx6+be/bGHP6fZHXoNPRqOnWKqtlz80yxeUoeCI3XrHJF8VUrKJEScN2q23ovRBvWS2W+nNgTrO/iyH9ttJVxoPRypm1FYUBqOicjjjPrC2ztZVJZne6yKzUBMr+q1AzNAT0Db6XWHMcAJdZIPRusxm2ViIhJ6zE/i8zCvkcaMI+Dt18gCJg85w52LN7iokAuh0YTFf6jQG2fLTDhwkEbOzL5LI7e6CpKeLvqM0j+I2FORzEJP6kk/QYKlm1baynUrVQ1tS6fQfF1A5Z0dGaxCozmWrtHQonVfH1Ob0TG+Q7oQUKw2GNOMPHcy1ETIs2+eSoCxWBzH1PrRrB1+//vvVq8u3VCDc4U7H5Q4GcHoBiX0QYBBgBwGW/IPAVm8IZObQ/wZR3/rWt771rW9961vf+ta3vvWtb33rW9/61re+9a1vfetb3/rWt771rW9961vf+ta3vvWtb3370u1/AVFHmnIAeAAA'
root=Path('/kaggle/working/wikirace-project')
with tarfile.open(fileobj=io.BytesIO(base64.b64decode(payload)), mode='r:gz') as archive:
    archive.extractall(root, filter='data')
(root/'data/wikirace').mkdir(parents=True,exist_ok=True)
for source in Path('/kaggle/input/laya-wikirace-data').iterdir():
    if source.is_file() and source.suffix in ('.json','.jsonl'):
        shutil.copy2(source,root/'data/wikirace'/source.name)
os.chdir(root)
print('Graph and splits unpacked')


In [ ]:
!PYTHONPATH=. OUTPUT_DIR=/kaggle/working/laya-wikirace python -u -m wikirace.train


In [ ]:
import json
from pathlib import Path
report=json.loads(Path('/kaggle/working/laya-wikirace/report.json').read_text())
print(json.dumps(report, indent=2))
